# Dataset Overview: Last 6 Months Sample (H&M)

This notebook performs EDA to identify what should be fixed in preprocessing.

Scope:
- Use transactions from the most recent 6 months in the dataset for the project sample window.
- Run EDA only on the first 300,000 rows from that 6-month subset for fast iteration.
- Check data quality and relational integrity across `transactions`, `customers`, and `articles`.


As gotten from https://www.kaggle.com/competitions/h-and-m-personalized-fashion-recommendations/data:

Files
images/ - a folder of images corresponding to each article_id; images are placed in subfolders starting with the first three digits of the article_id; note, not all article_id values have a corresponding image.

articles.csv - detailed metadata for each article_id available for purchase

customers.csv - metadata for each customer_id in dataset

sample_submission.csv - Ignored

transactions_train.csv - the training data, consisting of the purchases each customer for each date, as well as additional information. Duplicate rows correspond to multiple purchases of the same item. 

Out of the 6 months of training, the last week will be used for evaluation. 


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

CANDIDATES = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
PROJECT_ROOT = next((p for p in CANDIDATES if (p / "data" / "raw").exists()), Path.cwd())

RAW_DIR = PROJECT_ROOT / "data" / "raw"
TX_PATH = RAW_DIR / "transactions_train.csv"
CUSTOMERS_PATH = RAW_DIR / "customers.csv"
ARTICLES_PATH = RAW_DIR / "articles.csv"

print(f"Project root: {PROJECT_ROOT}")
for p in [TX_PATH, CUSTOMERS_PATH, ARTICLES_PATH]:
    print(f"{p}: {'OK' if p.exists() else 'MISSING'}")

Project root: /Users/chikire/mds/Reco-Nova
/Users/chikire/mds/Reco-Nova/data/raw/transactions_train.csv: OK
/Users/chikire/mds/Reco-Nova/data/raw/customers.csv: OK
/Users/chikire/mds/Reco-Nova/data/raw/articles.csv: OK


In [2]:
#tx_cols = ["t_dat", "customer_id", "article_id", "price", "sales_channel_id"]

transactions = pd.read_csv(TX_PATH)
customers = pd.read_csv(CUSTOMERS_PATH)
articles = pd.read_csv(ARTICLES_PATH)

transactions["t_dat"] = pd.to_datetime(transactions["t_dat"], errors="coerce")
transactions["article_id"] = transactions["article_id"].astype("string")
transactions["customer_id"] = transactions["customer_id"].astype("string")

print("Shapes")
print("transactions:", transactions.shape)
print("customers   :", customers.shape)
print("articles    :", articles.shape)

print("\nDate range")
print(transactions["t_dat"].min(), "->", transactions["t_dat"].max())

Shapes
transactions: (31788324, 5)
customers   : (1371980, 7)
articles    : (105542, 25)

Date range
2018-09-20 00:00:00 -> 2020-09-22 00:00:00


In [3]:
transactions.head(5)

,t_dat,customer_id,article_id,price,sales_channel_id
0,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,663713001,0.050831,2
1,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,541518023,0.030492,2
2,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,505221004,0.015237,2
3,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,685687003,0.016932,2
4,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,685687004,0.016932,2


In [4]:
customers.head(5)

,customer_id,FN,Active,club_member_status,fashion_news_frequency,age,postal_code
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,NaN,NaN,ACTIVE,NONE,49.0,52043ee2162cf5aa7ee79974281641c6f11a68d276429a...
1,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,NaN,NaN,ACTIVE,NONE,25.0,2973abc54daa8a5f8ccfe9362140c63247c5eee03f1d93...
2,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,NaN,NaN,ACTIVE,NONE,24.0,64f17e6a330a85798e4998f62d0930d14db8db1c054af6...
3,00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2...,NaN,NaN,ACTIVE,NONE,54.0,5d36574f52495e81f019b680c843c443bd343d5ca5b1c2...
4,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...,1.0,1.0,ACTIVE,Regularly,52.0,25fa5ddee9aac01b35208d01736e57942317d756b32ddd...


In [5]:
articles.head(5)

,article_id,product_code,prod_name,product_type_no,product_type_name,product_group_name,graphical_appearance_no,graphical_appearance_name,colour_group_code,colour_group_name,perceived_colour_value_id,perceived_colour_value_name,perceived_colour_master_id,perceived_colour_master_name,department_no,department_name,index_code,index_name,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name,detail_desc
0,108775015,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,9,Black,4,Dark,5,Black,1676,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
1,108775044,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,10,White,3,Light,9,White,1676,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
2,108775051,108775,Strap top (1),253,Vest top,Garment Upper body,1010017,Stripe,11,Off White,1,Dusty Light,9,White,1676,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
3,110065001,110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,9,Black,4,Dark,5,Black,1339,Clean Lingerie,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear","Microfibre T-shirt bra with underwired, moulde..."
4,110065002,110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,10,White,3,Light,9,White,1339,Clean Lingerie,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear","Microfibre T-shirt bra with underwired, moulde..."


In [6]:
transactions.columns.tolist()

['t_dat', 'customer_id', 'article_id', 'price', 'sales_channel_id']

In [7]:
customers.columns.tolist()

['customer_id',
 'FN',
 'Active',
 'club_member_status',
 'fashion_news_frequency',
 'age',
 'postal_code']

In [8]:
articles.columns.tolist()

['article_id',
 'product_code',
 'prod_name',
 'product_type_no',
 'product_type_name',
 'product_group_name',
 'graphical_appearance_no',
 'graphical_appearance_name',
 'colour_group_code',
 'colour_group_name',
 'perceived_colour_value_id',
 'perceived_colour_value_name',
 'perceived_colour_master_id',
 'perceived_colour_master_name',
 'department_no',
 'department_name',
 'index_code',
 'index_name',
 'index_group_no',
 'index_group_name',
 'section_no',
 'section_name',
 'garment_group_no',
 'garment_group_name',
 'detail_desc']

In [9]:
max_date = transactions["t_dat"].max()
cutoff_date = max_date - pd.DateOffset(months=6)

tx_last_6m = transactions.loc[transactions["t_dat"] >= cutoff_date].copy()

tx_last_6m = tx_last_6m.sort_values("t_dat").reset_index(drop=True)

sample_n = 300_000  # EDA is intentionally limited to first 300k rows.
tx_sample = tx_last_6m.head(sample_n).copy()
sample_frac = len(tx_sample) / max(len(tx_last_6m), 1)

print(f"max_date: {max_date}")
print(f"cutoff_date: {cutoff_date}")
print(f"rows in last 6 months: {len(tx_last_6m):,}")
print(f"rows in EDA sample (first rows): {len(tx_sample):,} ({sample_frac:.2%} of last 6 months)")

max_date: 2020-09-22 00:00:00
cutoff_date: 2020-03-22 00:00:00
rows in last 6 months: 8,185,912
rows in EDA sample (first rows): 300,000 (3.66% of last 6 months)


In [10]:
tx_sample.info()

<class 'pandas.DataFrame'>
RangeIndex: 300000 entries, 0 to 299999
Data columns (total 5 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   t_dat             300000 non-null  datetime64[us]
 1   customer_id       300000 non-null  string        
 2   article_id        300000 non-null  string        
 3   price             300000 non-null  float64       
 4   sales_channel_id  300000 non-null  int64         
dtypes: datetime64[us](1), float64(1), int64(1), string(2)
memory usage: 32.4 MB


In [11]:
def pct(x, total):
    return 100.0 * x / total if total else np.nan

n = len(tx_sample)

quality = {
    "rows": n,
    "missing_t_dat": int(tx_sample["t_dat"].isna().sum()),
    "missing_customer_id": int(tx_sample["customer_id"].isna().sum()),
    "missing_article_id": int(tx_sample["article_id"].isna().sum()),
    "missing_price": int(tx_sample["price"].isna().sum()),
    "missing_sales_channel_id": int(tx_sample["sales_channel_id"].isna().sum()),
    "duplicate_rows": int(tx_sample.duplicated().sum()),
    "non_positive_price": int((tx_sample["price"].fillna(-1) <= 0).sum()),
    "invalid_sales_channel": int((~tx_sample["sales_channel_id"].isin([1, 2])).fillna(True).sum()),
}

quality_df = pd.DataFrame(
    {
        "metric": list(quality.keys()),
        "count": list(quality.values()),
    }
)
quality_df["pct_of_sample"] = quality_df["count"].apply(lambda x: pct(x, n))
quality_df

,metric,count,pct_of_sample
0,rows,300000,100.000000
1,missing_t_dat,0,0.000000
2,missing_customer_id,0,0.000000
3,missing_article_id,0,0.000000
4,missing_price,0,0.000000
5,missing_sales_channel_id,0,0.000000
6,duplicate_rows,30797,10.265667
7,non_positive_price,0,0.000000
8,invalid_sales_channel,0,0.000000


In [12]:
customer_ids_master = set(customers["customer_id"].astype("string"))
article_ids_master = set(articles["article_id"].astype("string"))

orphans = {
    "customer_ids_not_in_customers": int((~tx_sample["customer_id"].isin(customer_ids_master)).sum()),
    "article_ids_not_in_articles": int((~tx_sample["article_id"].isin(article_ids_master)).sum()),
}

id_profile = pd.DataFrame(
    {
        "metric": [
            "unique_customers_in_sample",
            "unique_articles_in_sample",
            "customer_ids_not_in_customers",
            "article_ids_not_in_articles",
        ],
        "value": [
            int(tx_sample["customer_id"].nunique(dropna=True)),
            int(tx_sample["article_id"].nunique(dropna=True)),
            orphans["customer_ids_not_in_customers"],
            orphans["article_ids_not_in_articles"],
        ],
    }
)
id_profile

,metric,value
0,unique_customers_in_sample,67879
1,unique_articles_in_sample,16795
2,customer_ids_not_in_customers,0
3,article_ids_not_in_articles,0


In [13]:
daily_activity = (
    tx_last_6m.groupby(tx_last_6m["t_dat"].dt.to_period("M"))
    .size()
    .rename("transactions")
    .reset_index()
)
daily_activity["t_dat"] = daily_activity["t_dat"].astype(str)

daily_activity

,t_dat,transactions
0,2020-03,331745
1,2020-04,1340882
2,2020-05,1361815
3,2020-06,1764507
4,2020-07,1351502
5,2020-08,1237192
6,2020-09,798269


In [14]:
recommendations = []

if quality["missing_t_dat"] > 0:
    recommendations.append("Drop rows with invalid or missing t_dat.")
if quality["missing_customer_id"] > 0:
    recommendations.append("Drop rows with missing customer_id.")
if quality["missing_article_id"] > 0:
    recommendations.append("Drop rows with missing article_id.")
if quality["missing_price"] > 0 or quality["non_positive_price"] > 0:
    recommendations.append("Coerce price to numeric and remove rows where price <= 0 or missing.")
if quality["invalid_sales_channel"] > 0:
    recommendations.append("Restrict sales_channel_id to expected values {1, 2}; mark others invalid.")
if orphans["article_ids_not_in_articles"] > 0:
    recommendations.append("Drop transactions with article_id not found in articles metadata.")
if orphans["customer_ids_not_in_customers"] > 0:
    recommendations.append("Handle transactions with customer_id missing from customers table (drop or map to unknown).")

if not recommendations:
    recommendations.append("No critical issues found in this sample. Keep current preprocessing and add assert-based checks.")

pd.DataFrame({"preprocessing_actions": recommendations})

,preprocessing_actions
0,No critical issues found in this sample. Keep ...


## Data Quality: customers.csv & articles.csv

Check missing values, cardinality, numeric distributions, and anomalies for both files.

In [15]:
def profile_df(df: pd.DataFrame, name: str) -> pd.DataFrame:
    total = len(df)
    rows = []
    for col in df.columns:
        s = df[col]
        missing = int(s.isna().sum())
        unique = int(s.nunique(dropna=True))
        dtype = str(s.dtype)
        top_val = s.dropna().value_counts().index[0] if missing < total else None
        top_freq = int(s.value_counts(dropna=True).iloc[0]) if missing < total else 0
        rows.append({
            "column": col,
            "dtype": dtype,
            "missing": missing,
            "missing_%": round(100.0 * missing / total, 2),
            "unique": unique,
            "unique_%": round(100.0 * unique / total, 2),
            "top_value": str(top_val)[:50],
            "top_freq": top_freq,
        })
    result = pd.DataFrame(rows).sort_values("missing_%", ascending=False)
    print(f"\n=== {name}: {total:,} rows, {len(df.columns)} columns ===")
    return result


# Restrict to only customer/article IDs that appear in the 300k tx_sample.
sample_customer_ids = set(tx_sample["customer_id"].dropna().unique())
sample_article_ids = set(tx_sample["article_id"].dropna().unique())

customers_sample = customers[
    customers["customer_id"].astype("string").isin(sample_customer_ids)
].copy()
articles_sample = articles[
    articles["article_id"].astype("string").isin(sample_article_ids)
].copy()

print(f"customers in tx_sample: {len(customers_sample):,} / {len(customers):,} total")
print(f"articles  in tx_sample: {len(articles_sample):,} / {len(articles):,} total")

customers_profile = profile_df(customers_sample, "customers (tx_sample subset)")
customers_profile

customers in tx_sample: 67,879 / 1,371,980 total
articles  in tx_sample: 16,795 / 105,542 total

=== customers (tx_sample subset): 67,879 rows, 7 columns ===


,column,dtype,missing,missing_%,unique,unique_%,top_value,top_freq
2,Active,float64,42036,61.93,1,0.00,1.0,25843
1,FN,float64,41669,61.39,1,0.00,1.0,26210
5,age,float64,253,0.37,74,0.11,21.0,3673
4,fashion_news_frequency,str,146,0.22,3,0.00,NONE,41448
3,club_member_status,str,130,0.19,3,0.00,ACTIVE,65998
0,customer_id,str,0,0.00,67879,100.00,0001d44dbe7f6c4b35200abdb052c77a87596fe1bdcc37...,1
6,postal_code,str,0,0.00,58199,85.74,2c29ae653a9282cce4151bd87643c907644e09541abc28...,451


In [16]:
articles_profile = profile_df(articles_sample, "articles (tx_sample subset)")
articles_profile


=== articles (tx_sample subset): 16,795 rows, 25 columns ===


,column,dtype,missing,missing_%,unique,unique_%,top_value,top_freq
24,detail_desc,str,27,0.16,9381,55.86,Fine-knit trainer socks in a soft cotton blend...,25
13,perceived_colour_master_name,str,0,0.00,19,0.11,Black,4328
23,garment_group_name,str,0,0.00,21,0.13,Jersey Fancy,3065
22,garment_group_no,int64,0,0.00,21,0.13,1005,3065
21,section_name,str,0,0.00,53,0.32,Divided Collection,1835
20,section_no,int64,0,0.00,53,0.32,53,1835
19,index_group_name,str,0,0.00,5,0.03,Ladieswear,8607
18,index_group_no,int64,0,0.00,5,0.03,1,8607
17,index_name,str,0,0.00,10,0.06,Ladieswear,5791
16,index_code,str,0,0.00,10,0.06,A,5791


In [17]:
# Numeric distributions for customers in tx_sample
customers_numeric = customers_sample.select_dtypes(include="number")
if not customers_numeric.empty:
    display(customers_numeric.describe().T.round(2))

# Numeric distributions for articles in tx_sample
articles_numeric = articles_sample.select_dtypes(include="number")
if not articles_numeric.empty:
    display(articles_numeric.describe().T.round(2))

,count,mean,std,min,25%,50%,75%,max
FN,26210.0,1.00,0.00,1.0,1.0,1.0,1.0,1.0
Active,25843.0,1.00,0.00,1.0,1.0,1.0,1.0,1.0
age,67626.0,34.45,13.55,16.0,24.0,29.0,46.0,98.0


,count,mean,std,min,25%,50%,75%,max
article_id,16795.0,7.460501e+08,1.146430e+08,108775044.0,701772001.5,778224002.0,824878004.5,908625002.0
product_code,16795.0,7.460501e+05,1.146430e+05,108775.0,701772.0,778224.0,824878.0,908625.0
product_type_no,16795.0,2.439400e+02,6.760000e+01,-1.0,254.0,264.0,272.0,515.0
graphical_appearance_no,16795.0,1.009713e+06,1.742493e+04,-1.0,1010010.0,1010016.0,1010016.0,1010028.0
colour_group_code,16795.0,2.893000e+01,2.676000e+01,-1.0,9.0,12.0,51.0,93.0
perceived_colour_value_id,16795.0,3.110000e+00,1.520000e+00,-1.0,2.0,3.0,4.0,7.0
perceived_colour_master_id,16795.0,7.720000e+00,5.180000e+00,-1.0,4.0,5.0,11.0,20.0
department_no,16795.0,3.564100e+03,2.534960e+03,1201.0,1636.0,1941.0,5686.0,9989.0
index_group_no,16795.0,2.660000e+00,4.380000e+00,1.0,1.0,1.0,3.0,26.0
section_no,16795.0,3.930000e+01,2.358000e+01,2.0,15.0,46.0,60.0,97.0


In [18]:
# High-missing columns that may need imputation or dropping
MISSING_THRESHOLD = 20.0  # percent

cust_high_missing = customers_profile[
    customers_profile["missing_%"] > MISSING_THRESHOLD
][["column", "missing", "missing_%"]]

art_high_missing = articles_profile[
    articles_profile["missing_%"] > MISSING_THRESHOLD
][["column", "missing", "missing_%"]]

print(f"Customers columns with >{MISSING_THRESHOLD}% missing:")
display(cust_high_missing if not cust_high_missing.empty else pd.DataFrame({"result": ["None"]}))

print(f"\nArticles columns with >{MISSING_THRESHOLD}% missing:")
display(art_high_missing if not art_high_missing.empty else pd.DataFrame({"result": ["None"]}))

Customers columns with >20.0% missing:


,column,missing,missing_%
2,Active,42036,61.93
1,FN,41669,61.39



Articles columns with >20.0% missing:


,result
0,None


## Field Selection for Retrieval

Goal: define the minimum, high-value columns needed for a fast retrieval stage (candidate generation), informed by the 6-month project sample and 300k-row EDA subset.

Schema alignment with preprocessing:
- `REQUIRED_INTERACTION_COLUMNS`: `customer_id`, `article_id`, `t_dat`, `price`, `sales_channel_id`
- `REQUIRED_ITEM_COLUMNS`: `article_id`, `prod_name`, `product_type_name`, `product_group_name`, `department_name`, `section_name`, `garment_group_name`, `detail_desc`
- `CATEGORY_COLUMNS`: `prod_name`, `product_type_name`, `product_group_name`, `department_name`, `section_name`, `garment_group_name`, `detail_desc`

Selected fields for retrieval in this notebook (aligned to preprocessing):
- Interactions: `customer_id`, `article_id`, `t_dat`, `price`, `sales_channel_id`
- Item metadata for retrieval features: `prod_name`, `product_type_name`, `product_group_name`, `department_name`, `section_name`, `garment_group_name`, `detail_desc`
- Customer metadata kept lightweight for retrieval slices: `age`, `fashion_news_frequency`, `club_member_status`

### Why these fields were selected

| Group | Field | Reason for selection | Retrieval usage |
|---|---|---|---|
| Interactions | `customer_id` | Stable user key needed to connect behavior to a user. | Build user history and user-item candidate sets. |
| Interactions | `article_id` | Stable item key needed to connect events to catalog metadata. | Build item popularity/co-occurrence candidates and joins. |
| Interactions | `t_dat` | Provides recency and seasonality signal. | Time-window filtering and recency-aware candidate generation. |
| Interactions | `price` | Lightweight preference signal for spending behavior. | Price-aware filtering/segmentation during candidate retrieval. |
| Interactions | `sales_channel_id` | Distinguishes online/offline behavior patterns. | Channel-aware retrieval candidates and constraints. |
| Item metadata | `prod_name` | High-signal short item name used for lexical/product identity cues. | Boost same-name or near-name item candidates in sparse cases. |
| Item metadata | `product_type_name` | Fine-grained product identity signal. | Same-type candidate expansion when retrieving related items. |
| Item metadata | `product_group_name` | Broader category context. | Backoff/fallback category retrieval when type is sparse. |
| Item metadata | `department_name` | Mid-level merchandising grouping. | Candidate grouping by department affinity. |
| Item metadata | `section_name` | Contextual catalog organization. | Section-based candidate filtering and expansion. |
| Item metadata | `garment_group_name` | Robust apparel grouping for similarity. | Additional category-level fallback for retrieval coverage. |
| Item metadata | `detail_desc` | Rich descriptive text signal for semantic similarity. | Text-based candidate expansion and fallback retrieval. |
| Customer metadata | `age` | Compact demographic signal with strong utility. | Age-band priors for candidate ranking/retrieval. |
| Customer metadata | `fashion_news_frequency` | Proxy for engagement and preference intensity. | Tune candidate freshness/diversity by engagement profile. |
| Customer metadata | `club_member_status` | Proxy for loyalty/intent differences. | Membership-aware candidate priors. |

Notes:
- Keep retrieval tables narrow and clean (drop rows failing ID/date/price validity checks).
- Leave richer text/image features to ranking or multimodal stages.

In [19]:
retrieval_tx_fields = ["customer_id", "article_id", "t_dat", "price", "sales_channel_id"]
retrieval_item_fields = [
    "article_id",
    "prod_name",
    "product_type_name",
    "product_group_name",
    "department_name",
    "section_name",
    "garment_group_name",
    "detail_desc",
]
retrieval_customer_fields = [
    "customer_id",
    "age",
    "fashion_news_frequency",
    "club_member_status",
]

# Keep only fields that exist in source tables to make this cell robust.
retrieval_item_fields = [c for c in retrieval_item_fields if c in articles.columns]
retrieval_customer_fields = [c for c in retrieval_customer_fields if c in customers.columns]

retrieval_tx = tx_last_6m[retrieval_tx_fields].copy()
retrieval_items = articles[retrieval_item_fields].copy()
retrieval_customers = customers[retrieval_customer_fields].copy()

# Apply core quality filters from EDA findings.
retrieval_tx["t_dat"] = pd.to_datetime(retrieval_tx["t_dat"], errors="coerce")
retrieval_tx["price"] = pd.to_numeric(retrieval_tx["price"], errors="coerce")
retrieval_tx = retrieval_tx.dropna(subset=["customer_id", "article_id", "t_dat", "price"])
retrieval_tx = retrieval_tx[retrieval_tx["price"] > 0]
retrieval_tx = retrieval_tx[retrieval_tx["sales_channel_id"].isin([1, 2])]
retrieval_tx = retrieval_tx.drop_duplicates()

summary = pd.DataFrame(
    {
        "table": ["retrieval_tx", "retrieval_items", "retrieval_customers"],
        "rows": [len(retrieval_tx), len(retrieval_items), len(retrieval_customers)],
        "columns": [
            ", ".join(retrieval_tx.columns),
            ", ".join(retrieval_items.columns),
            ", ".join(retrieval_customers.columns),
        ],
    }
)

summary

,table,rows,columns
0,retrieval_tx,7450387,"customer_id, article_id, t_dat, price, sales_c..."
1,retrieval_items,105542,"article_id, prod_name, product_type_name, prod..."
2,retrieval_customers,1371980,"customer_id, age, fashion_news_frequency, club..."


## Behavior, Popularity, Sparsity, and Cold-Start Risk

This section quantifies user behavior, item popularity, interaction sparsity, and cold-start risk from the 300k transaction sample to guide model choices.

In [20]:
# 1) Analyze user activity distribution
user_activity = (
    tx_sample.groupby("customer_id").size().rename("n_interactions")
)

user_activity_summary = pd.DataFrame(
    {
        "metric": ["users", "mean", "median", "p90", "p95", "p99", "max"],
        "value": [
            int(user_activity.shape[0]),
            round(float(user_activity.mean()), 2),
            round(float(user_activity.median()), 2),
            round(float(user_activity.quantile(0.90)), 2),
            round(float(user_activity.quantile(0.95)), 2),
            round(float(user_activity.quantile(0.99)), 2),
            int(user_activity.max()),
        ],
    }
)

activity_bins = pd.cut(
    user_activity,
    bins=[0, 1, 2, 5, 10, 20, user_activity.max()],
    include_lowest=True,
)
user_activity_distribution = (
    activity_bins.value_counts().sort_index().rename_axis("interactions_per_user_bin")
    .reset_index(name="n_users")
)
user_activity_distribution["pct_users"] = (
    100 * user_activity_distribution["n_users"] / user_activity.shape[0]
).round(2)

print("User activity summary")
display(user_activity_summary)
print("\nUser activity distribution")
display(user_activity_distribution)

User activity summary


,metric,value
0,users,67879.00
1,mean,4.42
2,median,3.00
3,p90,9.00
4,p95,13.00
5,p99,22.00
6,max,98.00



User activity distribution


,interactions_per_user_bin,n_users,pct_users
0,"(-0.001, 1.0]",14353,21.14
1,"(1.0, 2.0]",14067,20.72
2,"(2.0, 5.0]",22133,32.61
3,"(5.0, 10.0]",12138,17.88
4,"(10.0, 20.0]",4393,6.47
5,"(20.0, 98.0]",795,1.17


In [21]:
# 2) Analyze item popularity distribution
item_popularity = (
    tx_sample.groupby("article_id").size().rename("n_interactions")
)

item_popularity_summary = pd.DataFrame(
    {
        "metric": ["items", "mean", "median", "p90", "p95", "p99", "max"],
        "value": [
            int(item_popularity.shape[0]),
            round(float(item_popularity.mean()), 2),
            round(float(item_popularity.median()), 2),
            round(float(item_popularity.quantile(0.90)), 2),
            round(float(item_popularity.quantile(0.95)), 2),
            round(float(item_popularity.quantile(0.99)), 2),
            int(item_popularity.max()),
        ],
    }
)

pop_bins = pd.cut(
    item_popularity,
    bins=[0, 1, 2, 5, 10, 20, item_popularity.max()],
    include_lowest=True,
)
item_popularity_distribution = (
    pop_bins.value_counts().sort_index().rename_axis("interactions_per_item_bin")
    .reset_index(name="n_items")
)
item_popularity_distribution["pct_items"] = (
    100 * item_popularity_distribution["n_items"] / item_popularity.shape[0]
).round(2)

print("Item popularity summary")
display(item_popularity_summary)
print("\nItem popularity distribution")
display(item_popularity_distribution)

Item popularity summary


,metric,value
0,items,16795.00
1,mean,17.86
2,median,4.00
3,p90,43.00
4,p95,77.00
5,p99,218.06
6,max,837.00



Item popularity distribution


,interactions_per_item_bin,n_items,pct_items
0,"(-0.001, 1.0]",4278,25.47
1,"(1.0, 2.0]",2208,13.15
2,"(2.0, 5.0]",3116,18.55
3,"(5.0, 10.0]",2223,13.24
4,"(10.0, 20.0]",1837,10.94
5,"(20.0, 837.0]",3133,18.65


In [22]:
# 3) Measure interaction matrix sparsity
n_interactions = len(tx_sample)
n_users = tx_sample["customer_id"].nunique()
n_items = tx_sample["article_id"].nunique()

matrix_size = n_users * n_items
density = n_interactions / matrix_size if matrix_size else 0.0
sparsity = 1.0 - density

sparsity_df = pd.DataFrame(
    {
        "metric": [
            "n_interactions",
            "n_users",
            "n_items",
            "matrix_size (users x items)",
            "density",
            "sparsity",
        ],
        "value": [
            int(n_interactions),
            int(n_users),
            int(n_items),
            int(matrix_size),
            round(float(density), 8),
            round(float(sparsity), 8),
        ],
    }
)

sparsity_df

,metric,value
0,n_interactions,3.000000e+05
1,n_users,6.787900e+04
2,n_items,1.679500e+04
3,matrix_size (users x items),1.140028e+09
4,density,2.631500e-04
5,sparsity,9.997369e-01


In [23]:
# 4) Identify cold-start risks for users/items
# Use the last 7 days of tx_sample as a proxy holdout to estimate unseen IDs.
split_point = tx_sample["t_dat"].max() - pd.Timedelta(days=7)
train_proxy = tx_sample[tx_sample["t_dat"] <= split_point].copy()
test_proxy = tx_sample[tx_sample["t_dat"] > split_point].copy()

train_users = set(train_proxy["customer_id"].dropna().unique())
train_items = set(train_proxy["article_id"].dropna().unique())
test_users = set(test_proxy["customer_id"].dropna().unique())
test_items = set(test_proxy["article_id"].dropna().unique())

new_users_in_test = test_users - train_users
new_items_in_test = test_items - train_items

cold_start_df = pd.DataFrame(
    {
        "metric": [
            "train_rows",
            "test_rows",
            "test_unique_users",
            "test_unique_items",
            "new_users_in_test",
            "new_items_in_test",
            "new_user_rate_in_test_users_%",
            "new_item_rate_in_test_items_%",
        ],
        "value": [
            int(len(train_proxy)),
            int(len(test_proxy)),
            int(len(test_users)),
            int(len(test_items)),
            int(len(new_users_in_test)),
            int(len(new_items_in_test)),
            round(100 * len(new_users_in_test) / max(len(test_users), 1), 2),
            round(100 * len(new_items_in_test) / max(len(test_items), 1), 2),
        ],
    }
)

cold_start_df

,metric,value
0,train_rows,93295.00
1,test_rows,206705.00
2,test_unique_users,47999.00
3,test_unique_items,15136.00
4,new_users_in_test,44460.00
5,new_items_in_test,5654.00
6,new_user_rate_in_test_users_%,92.63
7,new_item_rate_in_test_items_%,37.35


In [24]:
# 5) Summarize EDA insights for model planning
insights = []

if user_activity.quantile(0.95) > 10:
    insights.append("User activity is heavy-tailed; include popularity priors and cap very heavy-user influence.")
else:
    insights.append("User activity is relatively balanced; collaborative signals should generalize more evenly.")

if item_popularity.quantile(0.95) > 10:
    insights.append("Item popularity is heavy-tailed; combine personalized retrieval with popularity backfill.")
else:
    insights.append("Item popularity is less concentrated; broader candidate exploration is viable.")

if sparsity > 0.99:
    insights.append("Interaction matrix is extremely sparse; favor retrieval + ranking over dense neighborhood methods.")
else:
    insights.append("Sparsity is moderate; matrix/neighborhood methods are more feasible.")

if len(new_users_in_test) > 0:
    insights.append("User cold-start exists in holdout; include customer-feature and popularity fallback logic.")
if len(new_items_in_test) > 0:
    insights.append("Item cold-start exists in holdout; include content-based item features (`item_text`/metadata).")

plan_df = pd.DataFrame(
    {
        "checklist_item": [
            "User activity distribution",
            "Item popularity distribution",
            "Interaction matrix sparsity",
            "Cold-start risk",
            "Model planning summary",
        ],
        "status": ["Done", "Done", "Done", "Done", "Done"],
    }
)

insight_df = pd.DataFrame({"eda_insight_for_modeling": insights})

display(plan_df)
insight_df

,checklist_item,status
0,User activity distribution,Done
1,Item popularity distribution,Done
2,Interaction matrix sparsity,Done
3,Cold-start risk,Done
4,Model planning summary,Done


,eda_insight_for_modeling
0,User activity is heavy-tailed; include popular...
1,Item popularity is heavy-tailed; combine perso...
2,Interaction matrix is extremely sparse; favor ...
3,User cold-start exists in holdout; include cus...
4,Item cold-start exists in holdout; include con...
